# 93 — Re-evaluate / close M5 proof obligations

PRE_M6 候補の open obligations を **正当な RIGOROUS bound** で再評価する。

- j2_max 対応の projector cover / RSVD residual
- live M4→M3→M2 lineage parents（shared M2）
- audit checkpoint を ckpt_000014 固定から解放
- **CERTIFIED は出さない**。閉じられない義務は BLOCKED_MATH のまま
- production M6 (81) は `ONE_STEP_CERTIFIED` まで禁止


In [ ]:
from pathlib import Path
import os, sys, json

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
]
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'src' / 'campaign_b' / 'close_obligations.py').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Pull latest main; src/campaign_b/close_obligations.py not found.')
PERSIST_ROOT = Path(
    os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')
).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['VALIDATED_RG_PERSIST_ROOT'] = str(PERSIST_ROOT)
os.environ.setdefault('VALIDATED_RG_PERSIST_ACK', 'I_CONFIRM_THIS_PATH_IS_PERSISTENT')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT', PROJECT_ROOT)
print('PERSIST_ROOT', PERSIST_ROOT)


In [ ]:
from src.runtime import validate_persistent_root
from src.campaign_b.close_obligations import (
    list_obligation_queue, run_close_obligations_batch,
)

validate_persistent_root()
MAX_PACKAGES = 4
MAX_QUEUE = 50
ONLY_CAMPAIGN = None  # e.g. 'M7-20260720T152525Z-b-d87cf59a77dc'

QUEUE = list_obligation_queue(
    PERSIST_ROOT,
    max_candidates=MAX_QUEUE,
    only_campaign_run_id=ONLY_CAMPAIGN,
)
print(json.dumps({
    'queue_size': len(QUEUE),
    'top': [
        {
            'candidate_id': r.get('candidate_id'),
            'q_upper': r.get('q_upper'),
            'open_obligations': r.get('open_obligations'),
            'm5_run_id': r.get('m5_run_id'),
        }
        for r in QUEUE[:10]
    ],
}, indent=2, ensure_ascii=False, default=str))


In [ ]:
SESSION = run_close_obligations_batch(
    persistent_root=PERSIST_ROOT,
    project_root=PROJECT_ROOT,
    max_packages=MAX_PACKAGES,
    max_queue=MAX_QUEUE,
    only_campaign_run_id=ONLY_CAMPAIGN,
)
print(json.dumps({
    'session_id': SESSION.get('session_id'),
    'queue_size': SESSION.get('queue_size'),
    'attempted': SESSION.get('attempted'),
    'all_closed_count': SESSION.get('all_closed_count'),
    'm5_complete_count': SESSION.get('m5_complete_count'),
    'still_open': SESSION.get('still_open'),
    'errors': SESSION.get('errors'),
    'ledger': str(PERSIST_ROOT / 'campaign_b' / '_obligations' / 'LATEST_OBLIGATION_SESSION.json'),
    'certification_status': SESSION.get('certification_status'),
    'note': SESSION.get('note'),
}, indent=2, ensure_ascii=False, default=str))
